In [1]:
from dotenv import load_dotenv
import os
import clickhouse_connect
import pandas as pd
from datetime import datetime


load_dotenv()

client = clickhouse_connect.get_client(
    host=os.getenv('CLICKHOUSE_HOST'),
    port=int(os.getenv('CLICKHOUSE_PORT', 8123)), 
    username=os.getenv('CLICKHOUSE_USERNAME'),
    password=os.getenv('CLICKHOUSE_PASSWORD'),
    database=os.getenv('CLICKHOUSE_DATABASE')
)

In [2]:
# shipment_dataset_query = """
# SELECT 
#     -- 📦 Core dates (converted)
#     parseDateTimeBestEffortOrNull(b.shipped_date) AS shipped_date,
#     parseDateTimeBestEffortOrNull(b.eta) AS eta,
#     parseDateTimeBestEffortOrNull(b.receipt_date) AS receipt_date,

#     -- ⏱ Derived timing features
#     round(GREATEST(
#         dateDiff('day', 
#             toDate(parseDateTimeBestEffortOrNull(b.eta)), 
#             toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
#         ), 
#     0), 2) AS delay_days,

#     round(dateDiff('day', 
#         toDate(parseDateTimeBestEffortOrNull(b.shipped_date)), 
#         toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
#     ), 2) AS shipping_duration_days,

#     round(dateDiff('day', 
#         toDate(parseDateTimeBestEffortOrNull(p.purchase_order_date)), 
#         toDate(parseDateTimeBestEffortOrNull(b.shipped_date))
#     ), 2) AS lead_time_days,

#     (parseDateTimeBestEffortOrNull(b.receipt_date) < parseDateTimeBestEffortOrNull(b.eta)) AS is_early_delivery,

#     -- 🌍 Shipping details
#     b.coo AS coo,
#     b.scac AS scac,
#     round(toFloat64OrNull(b.tariff_amount), 2) AS tariff_amount,
#     round(toFloat64OrNull(b.ocean_freight), 2) AS ocean_freight,
#     p.delivery_terms AS delivery_terms,
#     p.shipment_terms AS po_shipment_terms,
#     p.tariff_type AS tariff_type,

#     -- 💰 Cost metrics
#     p.total_bcy AS total_bcy,

#     -- 📊 Product info
#     round(toFloat64OrNull(bni.quantity_in), 2) AS quantity_in,
#     i.sku AS item_sku,
#     i.brand AS item_brand,
#     i.manufacturer AS item_manufacturer,
#     i.product_category AS item_product_category,
#     i.size AS item_size,

#     -- 🏢 Vendor info
#     v.vendor_name AS vendor_name,

#     -- 🧮 🔁 Vendor-level aggregates (joined)
#     round(vs.vendor_avg_delay_days, 2) AS vendor_avg_delay_days,
#     vs.vendor_shipments,
#     round(vs.vendor_on_time_rate, 2) AS vendor_on_time_rate,
#     round(vs.vendor_p50_delay_days, 2) AS vendor_p50_delay_days,
#     round(vs.vendor_p90_delay_days, 2) AS vendor_p90_delay_days

# FROM zoho_books_analytics.batch_number_in AS bni
# INNER JOIN zoho_books_analytics.bills AS b 
#     ON bni.bill_id = b.bill_id 
# INNER JOIN zoho_books_analytics.bill_item AS bi 
#     ON b.bill_id = bi.bill_id
# INNER JOIN zoho_books_analytics.purchase_orders AS p 
#     ON b.purchase_order = p.purchase_order_number  
# INNER JOIN zoho_books_analytics.items AS i 
#     ON bi.product_id = i.item_id 
# INNER JOIN zoho_books_analytics.sales_orders AS so 
#     ON p.reference_number = so.sales_order 
# INNER JOIN zoho_books_analytics.customers AS c 
#     ON c.customer_id = so.customer_id
# INNER JOIN zoho_books_analytics.customer_item_mapping AS ci 
#     ON i.sku = ci.az_sku 
# INNER JOIN zoho_books_analytics.vendors AS v 
#     ON v.vendor_id = b.vendor_id

# /* ✅ Inline vendor-level aggregate */
# LEFT JOIN
# (
#     SELECT
#         v.vendor_id AS vendor_id,
#         round(avg(GREATEST(
#                 dateDiff('day',
#                     toDate(parseDateTimeBestEffortOrNull(b.eta)),
#                     toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
#                 ), 0)), 2) AS vendor_avg_delay_days,
#         count() AS vendor_shipments,
#         round(avg(GREATEST(
#                 dateDiff('day',
#                     toDate(parseDateTimeBestEffortOrNull(b.eta)),
#                     toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
#                 ), 0) = 0), 2) AS vendor_on_time_rate,
#         round(quantileExact(0.5)(GREATEST(
#                 dateDiff('day',
#                     toDate(parseDateTimeBestEffortOrNull(b.eta)),
#                     toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
#                 ), 0)), 2) AS vendor_p50_delay_days,
#         round(quantileExact(0.9)(GREATEST(
#                 dateDiff('day',
#                     toDate(parseDateTimeBestEffortOrNull(b.eta)),
#                     toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
#                 ), 0)), 2) AS vendor_p90_delay_days
#     FROM zoho_books_analytics.bills AS b
#     INNER JOIN zoho_books_analytics.purchase_orders AS p
#         ON b.purchase_order = p.purchase_order_number
#     INNER JOIN zoho_books_analytics.sales_orders AS so
#         ON p.reference_number = so.sales_order
#     INNER JOIN zoho_books_analytics.customers AS c
#         ON c.customer_id = so.customer_id
#     INNER JOIN zoho_books_analytics.vendors AS v
#         ON v.vendor_id = b.vendor_id
#     WHERE
#         c.customer_name LIKE 'Walmart%'
#         AND b.receipt_date != ''
#         AND b.shipped_date IS NOT NULL
#     GROUP BY v.vendor_id
# ) AS vs
#     ON vs.vendor_id = v.vendor_id

# WHERE 
#     c.customer_name LIKE 'Walmart%' 
#     AND b.receipt_date != ''
#     AND b.shipped_date IS NOT NULL
# ORDER BY bni.created_time DESC
# """

In [3]:
shipment_dataset_query = """
SELECT 
    -- 📦 Core dates (converted; no receipt_date used for row-level features)
    parseDateTimeBestEffortOrNull(b.shipped_date) AS shipped_date,
    parseDateTimeBestEffortOrNull(b.eta)          AS eta,
    round(GREATEST(
        dateDiff('day', 
             toDate(parseDateTimeBestEffortOrNull(b.eta)), 
            toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
        ), 
    0), 2) AS delay_days,

    -- ⏩ Scoring-safe timing features
    toFloat64(dateDiff('day', toDate(shipped_date), toDate(eta)))   AS promised_transit_days,
    toFloat64(dateDiff('day', toDate(now()),       toDate(eta)))    AS days_until_eta,        -- negative if past ETA
    toFloat64(dateDiff('day', toDate(shipped_date), toDate(now()))) AS days_since_ship_so_far,
    toFloat64(dateDiff('day', 
        toDate(parseDateTimeBestEffortOrNull(p.purchase_order_date)), 
        toDate(shipped_date)
    )) AS lead_time_days,

    -- 🌍 Shipping details
    b.coo                                           AS coo,
    b.scac                                          AS scac,
    round(toFloat64OrNull(b.tariff_amount), 2)      AS tariff_amount,
    round(toFloat64OrNull(b.ocean_freight), 2)      AS ocean_freight,
    p.delivery_terms                                AS delivery_terms,
    p.shipment_terms                                AS po_shipment_terms,
    p.tariff_type                                   AS tariff_type,

    -- 💰 Cost metrics
    p.total_bcy                                     AS total_bcy,

    -- 📊 Product info
    round(toFloat64OrNull(bni.quantity_in), 2)      AS quantity_in,
    i.sku                                           AS item_sku,
    i.brand                                         AS item_brand,
    i.manufacturer                                  AS item_manufacturer,
    i.product_category                              AS item_product_category,
    i.size                                          AS item_size,

    -- 🏢 Vendor info
    v.vendor_name                                   AS vendor_name,

    -- 🧮 Vendor-level aggregates (historical)
    -- Promised (ship→ETA): usable for all shipments
    round(vs.vendor_avg_promised_transit_days, 2)   AS vendor_avg_promised_transit_days,
    round(vs.vendor_p50_promised_transit_days, 2)   AS vendor_p50_promised_transit_days,
    round(vs.vendor_p90_promised_transit_days, 2)   AS vendor_p90_promised_transit_days,

    -- Realized (ETA→Receipt delay) — uses only rows with non-empty receipt_date
    round(vs.vendor_avg_realized_delay_days, 2)     AS vendor_avg_realized_delay_days,
    round(vs.vendor_p50_realized_delay_days, 2)     AS vendor_p50_realized_delay_days,
    round(vs.vendor_p90_realized_delay_days, 2)     AS vendor_p90_realized_delay_days,
    round(vs.vendor_on_time_rate, 2)                AS vendor_on_time_rate,
    vs.vendor_shipments_with_receipt                AS vendor_shipments_with_receipt

FROM zoho_books_analytics.batch_number_in AS bni
INNER JOIN zoho_books_analytics.bills AS b 
    ON bni.bill_id = b.bill_id 
INNER JOIN zoho_books_analytics.bill_item AS bi 
    ON b.bill_id = bi.bill_id
INNER JOIN zoho_books_analytics.purchase_orders AS p 
    ON b.purchase_order = p.purchase_order_number  
INNER JOIN zoho_books_analytics.items AS i 
    ON bi.product_id = i.item_id 
INNER JOIN zoho_books_analytics.sales_orders AS so 
    ON p.reference_number = so.sales_order 
INNER JOIN zoho_books_analytics.customers AS c 
    ON c.customer_id = so.customer_id
INNER JOIN zoho_books_analytics.customer_item_mapping AS ci 
    ON i.sku = ci.az_sku 
INNER JOIN zoho_books_analytics.vendors AS v 
    ON v.vendor_id = b.vendor_id

/* ✅ Vendor aggregates from history:
   - Promised transit: ship→ETA
   - Realized delay: max(ETA→Receipt, 0)
   Only include rows with non-empty receipt_date for realized metrics. */
LEFT JOIN
(
    SELECT
        v.vendor_id AS vendor_id,

        /* Promised (ship→ETA) */
        avg(toFloat64(dateDiff('day',
            toDate(parseDateTimeBestEffortOrNull(b.shipped_date)),
            toDate(parseDateTimeBestEffortOrNull(b.eta))
        ))) AS vendor_avg_promised_transit_days,

        quantileExact(0.5)(
            toFloat64(dateDiff('day',
                toDate(parseDateTimeBestEffortOrNull(b.shipped_date)),
                toDate(parseDateTimeBestEffortOrNull(b.eta))
        ))) AS vendor_p50_promised_transit_days,

        quantileExact(0.9)(
            toFloat64(dateDiff('day',
                toDate(parseDateTimeBestEffortOrNull(b.shipped_date)),
                toDate(parseDateTimeBestEffortOrNull(b.eta))
        ))) AS vendor_p90_promised_transit_days,

        /* Realized (ETA→Receipt) — exclude NULL/empty receipt_date */
        avg(
            toFloat64(
                greatest(
                    dateDiff('day',
                        toDate(parseDateTimeBestEffortOrNull(b.eta)),
                        toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
                    ),
                0)
            )
        ) AS vendor_avg_realized_delay_days,

        quantileExact(0.5)(
            toFloat64(
                greatest(
                    dateDiff('day',
                        toDate(parseDateTimeBestEffortOrNull(b.eta)),
                        toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
                    ),
                0)
            )
        ) AS vendor_p50_realized_delay_days,

        quantileExact(0.9)(
            toFloat64(
                greatest(
                    dateDiff('day',
                        toDate(parseDateTimeBestEffortOrNull(b.eta)),
                        toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
                    ),
                0)
            )
        ) AS vendor_p90_realized_delay_days,

        /* On-time rate: delay == 0 */
        avg(
            (greatest(
                dateDiff('day',
                    toDate(parseDateTimeBestEffortOrNull(b.eta)),
                    toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
                ),
            0) = 0) :: UInt8
        ) AS vendor_on_time_rate,

        /* Count only rows with a usable receipt_date */
        count() AS vendor_shipments_with_receipt

    FROM zoho_books_analytics.bills AS b
    INNER JOIN zoho_books_analytics.purchase_orders AS p
        ON b.purchase_order = p.purchase_order_number
    INNER JOIN zoho_books_analytics.sales_orders AS so
        ON p.reference_number = so.sales_order
    INNER JOIN zoho_books_analytics.customers AS c
        ON c.customer_id = so.customer_id
    INNER JOIN zoho_books_analytics.vendors AS v
        ON v.vendor_id = b.vendor_id
    WHERE
        c.customer_name LIKE 'Walmart%'
        AND b.shipped_date IS NOT NULL
        AND b.eta IS NOT NULL
        AND b.receipt_date IS NOT NULL
        AND b.receipt_date != ''
    GROUP BY v.vendor_id
) AS vs
    ON vs.vendor_id = v.vendor_id

WHERE 
    c.customer_name LIKE 'Walmart%' 
    AND b.shipped_date IS NOT NULL
    AND b.eta IS NOT NULL
ORDER BY bni.created_time DESC
"""


In [4]:
result = client.query(shipment_dataset_query)

shipment_dataset_df = pd.DataFrame(result.result_rows, columns=[col for col in result.column_names])

In [5]:
shipment_dataset_df.columns

Index(['shipped_date', 'eta', 'delay_days', 'promised_transit_days',
       'days_until_eta', 'days_since_ship_so_far', 'lead_time_days', 'coo',
       'scac', 'tariff_amount', 'ocean_freight', 'delivery_terms',
       'po_shipment_terms', 'tariff_type', 'total_bcy', 'quantity_in',
       'item_sku', 'item_brand', 'item_manufacturer', 'item_product_category',
       'item_size', 'vendor_name', 'vendor_avg_promised_transit_days',
       'vendor_p50_promised_transit_days', 'vendor_p90_promised_transit_days',
       'vendor_avg_realized_delay_days', 'vendor_p50_realized_delay_days',
       'vendor_p90_realized_delay_days', 'vendor_on_time_rate',
       'vendor_shipments_with_receipt'],
      dtype='object')

## Simple EDA to reduce number of columns

In [6]:
shipment_dataset_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2424 entries, 0 to 2423
Data columns (total 30 columns):
 #   Column                            Non-Null Count  Dtype                  
---  ------                            --------------  -----                  
 0   shipped_date                      2424 non-null   datetime64[ns, Etc/UTC]
 1   eta                               2424 non-null   datetime64[ns, Etc/UTC]
 2   delay_days                        2424 non-null   int64                  
 3   promised_transit_days             2424 non-null   float64                
 4   days_until_eta                    2424 non-null   float64                
 5   days_since_ship_so_far            2424 non-null   float64                
 6   lead_time_days                    2424 non-null   float64                
 7   coo                               2424 non-null   object                 
 8   scac                              2424 non-null   object                 
 9   tariff_amount      

In [7]:
shipment_dataset_df.isna().sum()

shipped_date                           0
eta                                    0
delay_days                             0
promised_transit_days                  0
days_until_eta                         0
days_since_ship_so_far                 0
lead_time_days                         0
coo                                    0
scac                                   0
tariff_amount                       1683
ocean_freight                       1352
delivery_terms                         0
po_shipment_terms                      0
tariff_type                            0
total_bcy                              0
quantity_in                            0
item_sku                               0
item_brand                             0
item_manufacturer                      0
item_product_category                  0
item_size                              0
vendor_name                            0
vendor_avg_promised_transit_days       0
vendor_p50_promised_transit_days       0
vendor_p90_promi

In [8]:
shipment_dataset_df.fillna({"tariff_amount": 0, "ocean_freight": 0}, inplace=True)
shipment_dataset_df["tariff_type"].replace("", "Not Applicable", inplace=True)

/tmp/ipykernel_175195/650567830.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  shipment_dataset_df["tariff_type"].replace("", "Not Applicable", inplace=True)


In [9]:
shipment_dataset_df.head(n=5)

,shipped_date,eta,delay_days,promised_transit_days,days_until_eta,days_since_ship_so_far,lead_time_days,coo,scac,tariff_amount,...,item_size,vendor_name,vendor_avg_promised_transit_days,vendor_p50_promised_transit_days,vendor_p90_promised_transit_days,vendor_avg_realized_delay_days,vendor_p50_realized_delay_days,vendor_p90_realized_delay_days,vendor_on_time_rate,vendor_shipments_with_receipt
0,2025-10-28 00:00:00+00:00,2025-12-08 00:00:00+00:00,0,41.0,38.0,3.0,134.0,INDIA,CMDU,0.0,...,60/80,Aquatica Frozen Foods Global Pvt Ltd,57.52,51.0,74.0,4.78,4.0,6.0,0.0,104
1,2025-10-28 00:00:00+00:00,2025-12-08 00:00:00+00:00,0,41.0,38.0,3.0,134.0,INDIA,CMDU,0.0,...,60/80,Aquatica Frozen Foods Global Pvt Ltd,57.52,51.0,74.0,4.78,4.0,6.0,0.0,104
2,2025-10-28 00:00:00+00:00,2025-12-08 00:00:00+00:00,0,41.0,38.0,3.0,134.0,INDIA,CMDU,0.0,...,60/80,Aquatica Frozen Foods Global Pvt Ltd,57.52,51.0,74.0,4.78,4.0,6.0,0.0,104
3,2025-10-28 00:00:00+00:00,2025-12-08 00:00:00+00:00,0,41.0,38.0,3.0,134.0,INDIA,CMDU,0.0,...,60/80,Aquatica Frozen Foods Global Pvt Ltd,57.52,51.0,74.0,4.78,4.0,6.0,0.0,104
4,2025-10-13 00:00:00+00:00,2025-12-18 00:00:00+00:00,0,66.0,48.0,18.0,108.0,INDIA,MAEU,0.0,...,60/80,Neeli Aqua,64.06,62.0,74.0,4.44,5.0,6.0,0.0,18


In [10]:
shipment_dataset_df.to_csv('./results/raw_shipment_classification_dataset.csv', index=False)

## Extracting the inference data

In [11]:
inference_data_query = """ 
SELECT 
    -- 📦 Core dates (converted)
    parseDateTimeBestEffortOrNull(b.shipped_date) AS shipped_date,
    parseDateTimeBestEffortOrNull(b.eta) AS eta,
    parseDateTimeBestEffortOrNull(b.receipt_date) AS receipt_date,

    -- ⏱ Derived timing features
    round(GREATEST(
        dateDiff('day', 
            toDate(parseDateTimeBestEffortOrNull(b.eta)), 
            toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
        ), 
    0), 2) AS delay_days,

    round(dateDiff('day', 
        toDate(parseDateTimeBestEffortOrNull(b.shipped_date)), 
        toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
    ), 2) AS shipping_duration_days,

    round(dateDiff('day', 
        toDate(parseDateTimeBestEffortOrNull(p.purchase_order_date)), 
        toDate(parseDateTimeBestEffortOrNull(b.shipped_date))
    ), 2) AS lead_time_days,

    (parseDateTimeBestEffortOrNull(b.receipt_date) < parseDateTimeBestEffortOrNull(b.eta)) AS is_early_delivery,

    -- 🌍 Shipping details
    b.coo AS coo,
    b.scac AS scac,
    round(toFloat64OrNull(b.tariff_amount), 2) AS tariff_amount,
    round(toFloat64OrNull(b.ocean_freight), 2) AS ocean_freight,
    p.delivery_terms AS delivery_terms,
    p.shipment_terms AS po_shipment_terms,
    p.tariff_type AS tariff_type,

    -- 💰 Cost metrics
    p.total_bcy AS total_bcy,

    -- 📊 Product info
    round(toFloat64OrNull(bni.quantity_in), 2) AS quantity_in,
    i.sku AS item_sku,
    i.brand AS item_brand,
    i.manufacturer AS item_manufacturer,
    i.product_category AS item_product_category,
    i.size AS item_size,

    -- 🏢 Vendor info
    v.vendor_name AS vendor_name,

    -- 🧮 🔁 Vendor-level aggregates (joined)
    round(vs.vendor_avg_delay_days, 2) AS vendor_avg_delay_days,
    vs.vendor_shipments,
    round(vs.vendor_on_time_rate, 2) AS vendor_on_time_rate,
    round(vs.vendor_p50_delay_days, 2) AS vendor_p50_delay_days,
    round(vs.vendor_p90_delay_days, 2) AS vendor_p90_delay_days

FROM zoho_books_analytics.batch_number_in AS bni
INNER JOIN zoho_books_analytics.bills AS b 
    ON bni.bill_id = b.bill_id 
INNER JOIN zoho_books_analytics.bill_item AS bi 
    ON b.bill_id = bi.bill_id
INNER JOIN zoho_books_analytics.purchase_orders AS p 
    ON b.purchase_order = p.purchase_order_number  
INNER JOIN zoho_books_analytics.items AS i 
    ON bi.product_id = i.item_id 
INNER JOIN zoho_books_analytics.sales_orders AS so 
    ON p.reference_number = so.sales_order 
INNER JOIN zoho_books_analytics.customers AS c 
    ON c.customer_id = so.customer_id
INNER JOIN zoho_books_analytics.customer_item_mapping AS ci 
    ON i.sku = ci.az_sku 
INNER JOIN zoho_books_analytics.vendors AS v 
    ON v.vendor_id = b.vendor_id

/* ✅ Inline vendor-level aggregate */
LEFT JOIN
(
    SELECT
        v.vendor_id AS vendor_id,
        round(avg(GREATEST(
                dateDiff('day',
                    toDate(parseDateTimeBestEffortOrNull(b.eta)),
                    toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
                ), 0)), 2) AS vendor_avg_delay_days,
        count() AS vendor_shipments,
        round(avg(GREATEST(
                dateDiff('day',
                    toDate(parseDateTimeBestEffortOrNull(b.eta)),
                    toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
                ), 0) = 0), 2) AS vendor_on_time_rate,
        round(quantileExact(0.5)(GREATEST(
                dateDiff('day',
                    toDate(parseDateTimeBestEffortOrNull(b.eta)),
                    toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
                ), 0)), 2) AS vendor_p50_delay_days,
        round(quantileExact(0.9)(GREATEST(
                dateDiff('day',
                    toDate(parseDateTimeBestEffortOrNull(b.eta)),
                    toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
                ), 0)), 2) AS vendor_p90_delay_days
    FROM zoho_books_analytics.bills AS b
    INNER JOIN zoho_books_analytics.purchase_orders AS p
        ON b.purchase_order = p.purchase_order_number
    INNER JOIN zoho_books_analytics.sales_orders AS so
        ON p.reference_number = so.sales_order
    INNER JOIN zoho_books_analytics.customers AS c
        ON c.customer_id = so.customer_id
    INNER JOIN zoho_books_analytics.vendors AS v
        ON v.vendor_id = b.vendor_id
    WHERE
        c.customer_name LIKE 'Walmart%'
        AND b.shipped_date IS NOT NULL
    GROUP BY v.vendor_id
) AS vs
    ON vs.vendor_id = v.vendor_id

WHERE 
    c.customer_name LIKE 'Walmart%' 
    AND b.shipped_date IS NOT NULL
    AND bni.created_time BETWEEN '2025-08-01' AND '2025-10-01'
ORDER BY bni.created_time DESC
"""

In [12]:
result = client.query(inference_data_query)

inference_dataset_df = pd.DataFrame(result.result_rows, columns=[col for col in result.column_names])

In [13]:
inference_dataset_df.fillna({"tariff_amount": 0, "ocean_freight": 0}, inplace=True)
inference_dataset_df["tariff_type"].replace("", "Not Applicable", inplace=True)

/tmp/ipykernel_175195/4155879485.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  inference_dataset_df["tariff_type"].replace("", "Not Applicable", inplace=True)


In [14]:
inference_dataset_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 377 entries, 0 to 376
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype                  
---  ------                  --------------  -----                  
 0   shipped_date            377 non-null    datetime64[ns, Etc/UTC]
 1   eta                     377 non-null    datetime64[ns, Etc/UTC]
 2   receipt_date            233 non-null    datetime64[ns, Etc/UTC]
 3   delay_days              377 non-null    int64                  
 4   shipping_duration_days  233 non-null    float64                
 5   lead_time_days          377 non-null    int64                  
 6   is_early_delivery       233 non-null    float64                
 7   coo                     377 non-null    object                 
 8   scac                    377 non-null    object                 
 9   tariff_amount           377 non-null    float64                
 10  ocean_freight           377 non-null    float64               

In [15]:
date_time_columns = ["shipped_date", "eta", "receipt_date"]
for col in date_time_columns:
    inference_dataset_df[f"{col}"] = pd.to_datetime(
        inference_dataset_df[f"{col}"], errors="raise"
    )

inference_dataset_df["shipped_date_weekday"] = inference_dataset_df[
    "shipped_date"
].dt.weekday
inference_dataset_df["shipped_date_month"] = inference_dataset_df[
    "shipped_date"
].dt.month
inference_dataset_df["shipped_date_day"] = inference_dataset_df["shipped_date"].dt.day

inference_dataset_df.drop(columns=date_time_columns, inplace=True)

In [16]:
# 1. Remove 'USD' and any surrounding whitespace
inference_dataset_df['total_bcy'] = inference_dataset_df['total_bcy'].str.replace('USD', '').str.strip()

# 2. FIX: Remove all thousands separators (commas)
inference_dataset_df['total_bcy'] = inference_dataset_df['total_bcy'].str.replace(',', '')

# 3. Convert the clean string to float
inference_dataset_df['total_bcy'] = inference_dataset_df['total_bcy'].astype(float)

In [17]:
numeric_cols = [
    'delay_days',
    'shipping_duration_days', 'lead_time_days',
    'quantity_in', 'total_bcy', 'vendor_avg_delay_days', 
    'vendor_shipments', 'vendor_on_time_rate', 'vendor_p50_delay_days', 
    'vendor_p90_delay_days'
]


for col in numeric_cols:
    inference_dataset_df = inference_dataset_df[inference_dataset_df[col] >= 0]

In [18]:
# Convert total_bcy to numeric
inference_dataset_df['total_bcy'] = pd.to_numeric(inference_dataset_df['total_bcy'], errors='coerce')

In [19]:
distance_dict = {
    'INDIA': 11000,
    'CHINA': 6000,
    'INDONESIA': 8200,
    'VIETNAM': 6200,
    'ECUADOR': 2100,
    'THAILAND': 8100
}
inference_dataset_df['distance_nm'] = inference_dataset_df['coo'].map(distance_dict)
inference_dataset_df.drop(columns=['coo','item_product_category'], inplace=True)

In [20]:
inference_dataset_df.to_csv('./results/raw_shipment_classification_inference_dataset.csv', index=False)